# 04c — LangGraph: durable remediation control plane

## Scenario: a safe response to an ambiguous checkout incident

At **09:04**, European checkout conversion falls 31%. Service health is mostly green, a deployment completed at 08:42, and six enterprise customers have complained. The system may collect evidence, form a rollback proposal, and route work, but it must **never** execute a restart or rollback solely because a model suggested it.

This notebook builds the control plane around that boundary. It uses a deterministic, credential-free simulator so every exercise runs locally, then maps each decision to a real LangGraph feature. The aim is not to make a maximally autonomous agent. It is to make a system that can pause, recover, explain itself, and act only when policy permits it.

**You will learn**

- when a `StateGraph` is more useful than a simple agent loop;
- how state, nodes, edges, reducers, and conditional routing fit together;
- how checkpoints and thread IDs support durable execution;
- how to design interrupts for review, approval, modification, and rejection;
- how to separate thread memory from cross-thread memory;
- how to stream progress, retry safely, and enforce idempotent side effects; and
- how to combine model judgment with deterministic policy and evaluation.

> **Safety contract:** all “actions” in this lab are simulated. A production action must be authorized by an authenticated identity, bound to exact arguments, recorded, and executed through an idempotent service boundary.


## 1. The architecture at a glance

A graph makes the control flow inspectable. The model may help interpret logs or summarize evidence *inside* a node, but routing, approval, persistence, and side-effect boundaries are application code.

![LangGraph remediation lifecycle](../../../assets/langgraph-remediation-lifecycle.svg)

### Why this is a LangGraph-shaped problem

A linear function is enough for “read status, format report.” This incident can pause for a human, survive a process restart, branch on evidence quality, and resume later with the same state. Those requirements call for explicit orchestration.

| Requirement | Graph design response | Why a prompt alone is insufficient |
| --- | --- | --- |
| Evidence may be incomplete | conditional edge to escalation | a model can sound certain without enough evidence |
| A human may respond later | checkpoint + stable `thread_id` | a process-local conversation can disappear |
| The action is consequential | interrupt before execution | “ask for approval” in a prompt does not authenticate anyone |
| A node can run again | idempotency at the action boundary | retries can duplicate restarts, tickets, or notifications |
| Operators need visibility | streamed events and audit facts | a final answer hides the trajectory |

LangGraph is intentionally low level: it mixes deterministic and model-driven steps in a stateful graph, while leaving architecture and policy choices explicit. Read the [official overview](https://docs.langchain.com/oss/python/langgraph/overview) before adopting it for a small task.


## 2. Model the incident state before writing nodes

State is the contract between nodes. Keep it **small, typed, observable, and purpose-bound**. Store evidence IDs or summarized facts—not raw secrets, unrestricted model transcripts, or a permanent assertion that a particular component is “usually” at fault.

A useful design question for every field is: *who writes it, who reads it, how long should it live, and can an operator inspect or correct it?*

```python
from typing import Annotated, Literal, TypedDict
import operator

class IncidentState(TypedDict):
    incident_id: str
    request: str
    evidence: Annotated[list[dict], operator.add]  # append updates from nodes
    confidence: float
    suspected_cause: str | None
    proposal: dict | None
    approval: dict | None
    action_receipt: str | None
    status: Literal["triaging", "needs_evidence", "awaiting_approval", "completed", "escalated", "rejected"]
```

`Annotated[..., operator.add]` is a reducer: independent nodes can contribute evidence without one update replacing the other. Reducers are powerful, but do not use an append-only field as an unbounded transcript. In production, summarize, retain selectively, and set retention policies.

**State vs. memory**

- A **checkpointer** persists state snapshots for one thread. It supports interruption, recovery, and inspection.
- A **store** holds application-defined information across threads, such as validated operator preferences—not unverified incident hypotheses.

LangGraph documents this distinction in its [persistence guide](https://docs.langchain.com/oss/python/langgraph/persistence). In-memory checkpointing is useful for a notebook; production needs a durable, access-controlled backend and a deletion/retention policy.


In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd().resolve()
for candidate in (course_dir, *course_dir.parents):
    if (candidate / "curriculum" / "beginner" / "04-agent-development-frameworks").exists():
        course_dir = candidate / "curriculum" / "beginner" / "04-agent-development-frameworks"
        break
else:
    raise RuntimeError("Run this notebook from a checkout of the repository.")

if str(course_dir) not in sys.path:
    sys.path.insert(0, str(course_dir))

from lab import INCIDENT_EVIDENCE, langgraph_shaped_approval

incident = {
    "incident_id": "inc-eu-482",
    "request": "Investigate a 31% Europe checkout conversion drop after the 08:42 release.",
    "evidence": [],
    "confidence": 0.0,
    "suspected_cause": None,
    "proposal": None,
    "approval": None,
    "action_receipt": None,
    "status": "triaging",
}
incident


## 3. Step 1 — Build deterministic nodes and explicit routes

Start with deterministic fixtures. That makes the architecture testable before an LLM is introduced. A model can later generate an evidence summary or classify a log pattern, but it should return structured data that these nodes validate.

The node functions below follow a useful rule: **return a state update, not a hidden global mutation**. This is the mental model behind a `StateGraph` node.


In [ ]:
def collect_evidence(state: dict) -> dict:
    """Read-only collection. A real implementation would call scoped tools."""
    evidence = [
        {"id": item.source_id, "claim": item.claim, "confidence": item.confidence}
        for item in INCIDENT_EVIDENCE.values()
    ]
    return {"evidence": evidence, "status": "needs_evidence"}


def assess_evidence(state: dict) -> dict:
    """Deterministic stand-in for a validated analysis node."""
    evidence = state["evidence"]
    deployment_signal = any("release" in item["claim"] for item in evidence)
    checkout_signal = any("degraded" in item["claim"] for item in evidence)
    confidence = round(sum(item["confidence"] for item in evidence) / len(evidence), 2)
    cause = "recent checkout release likely contributed" if deployment_signal and checkout_signal else None
    return {"confidence": confidence, "suspected_cause": cause}


def route_after_assessment(state: dict) -> str:
    """The conditional edge: deterministic thresholds are policy, not model prose."""
    if state["confidence"] < 0.80 or not state["suspected_cause"]:
        return "escalate"
    return "prepare_proposal"


def prepare_proposal(state: dict) -> dict:
    return {
        "proposal": {
            "action": "restart_checkout",
            "arguments": {"service": "checkout", "region": "eu-west"},
            "incident_id": state["incident_id"],
            "evidence_ids": [item["id"] for item in state["evidence"]],
            "reason": "Evidence indicates a checkout degradation immediately after the release.",
        },
        "status": "awaiting_approval",
    }


def escalate(state: dict) -> dict:
    return {
        "status": "escalated",
        "proposal": None,
        "reason": "Evidence is insufficient for a production remediation proposal.",
    }

state = {**incident, **collect_evidence(incident)}
state.update(assess_evidence(state))
branch = route_after_assessment(state)
state.update(prepare_proposal(state) if branch == "prepare_proposal" else escalate(state))
{"route": branch, "state": state}


### The real `StateGraph` shape

The following is a **reference implementation sketch**. Install `langgraph` to run it; the deterministic cells in this notebook do not require it. Notice the division of responsibilities:

- Nodes read state and return updates.
- `add_conditional_edges` makes the decision boundary visible.
- A checkpointer is passed at compile time.
- The graph carries a `thread_id`; it is a persistent cursor, not an authorization credential.

```python
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(IncidentState)
builder.add_node("collect_evidence", collect_evidence)
builder.add_node("assess", assess_evidence)
builder.add_node("prepare_proposal", prepare_proposal)
builder.add_node("escalate", escalate)
builder.add_edge(START, "collect_evidence")
builder.add_edge("collect_evidence", "assess")
builder.add_conditional_edges(
    "assess",
    route_after_assessment,
    {"prepare_proposal": "prepare_proposal", "escalate": "escalate"},
)
builder.add_edge("prepare_proposal", END)
builder.add_edge("escalate", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "inc-eu-482"}}
result = graph.invoke(incident, config=config)
```

For a production service, replace `InMemorySaver`: it loses state at process restart. The [persistence documentation](https://docs.langchain.com/oss/python/langgraph/persistence) discusses durable checkpoint backends and retention.


## 4. Step 2 — Place the human approval boundary

The graph is allowed to **propose** a restart. It is not allowed to execute it. A policy layer determines whether an approval is required, then LangGraph pauses with an interrupt. The approval screen must show the exact action, arguments, evidence IDs, risk, expiry, and incident ID.

![Approval boundary](../../../assets/langgraph-approval-boundary.svg)

### Approval payload checklist

1. Authenticate the operator outside the model and record their identity.
2. Verify the operator’s role, tenant, on-call responsibility, and separation-of-duties policy.
3. Bind approval to an immutable action digest: changing arguments invalidates approval.
4. Set an expiry; do not resume a week-old emergency action without re-review.
5. Allow **approve**, **reject**, and **modify/re-plan**—not just a Boolean.
6. Treat the resume payload as untrusted input and validate it.
7. Audit the proposal, evidence, approver, decision, timestamp, and execution receipt.

LangGraph’s [interrupt documentation](https://docs.langchain.com/oss/python/langgraph/interrupts) explains that an interrupt saves graph state and resumes through `Command(resume=...)` using the same thread ID. It also warns that the node is restarted when resuming, so side effects that occur before an interrupt must be idempotent.


In [ ]:
proposal_only = langgraph_shaped_approval("prepare_rollback")
paused = langgraph_shaped_approval("restart_checkout")
approved = langgraph_shaped_approval("restart_checkout", approved=True)

{"proposal_only": proposal_only, "paused": paused, "approved": approved}


### Real interrupt pattern (optional SDK code)

```python
from langgraph.types import Command, interrupt


def approval_node(state: IncidentState):
    proposal = state["proposal"]
    decision = interrupt({
        "kind": "remediation_approval",
        "incident_id": state["incident_id"],
        "proposal": proposal,
        "expires_at": "2026-08-10T09:30:00Z",
    })
    # Validate this schema, identity, expiry, and action digest in application code.
    if decision["decision"] == "reject":
        return {"approval": decision, "status": "rejected"}
    if decision["decision"] != "approve":
        return {"approval": decision, "status": "needs_evidence"}
    return {"approval": decision, "status": "approved"}

# First call pauses. A durable checkpointer and same thread ID are required.
graph.invoke(initial_state, config=config)
# Resume only after server-side authorization checks.
graph.invoke(Command(resume={"decision": "approve", "approver_id": "oncall-17"}), config=config)
```

Do not trust `approver_id` supplied by a browser client. Derive it from the authenticated server session, and perform authorization before calling `Command(resume=...)`.


## 5. Step 3 — Make retries safe: idempotency, recovery, and error routes

Durability means a graph can retry or resume. It does **not** make external effects safe by itself. Put the side effect behind a service that accepts a unique idempotency key and returns the original receipt on repetition.

| Failure | Correct response | Unsafe response |
| --- | --- | --- |
| log query times out | retry bounded read with backoff | retry forever or treat timeout as proof |
| approval expired | return to proposal/review | execute because the old approval was once valid |
| execution response lost | retry with same idempotency key | send a second restart blindly |
| permission denied | stop and escalate | ask the model to find a way around it |
| model output malformed | validate and re-plan/escalate | coerce it into an action |

Keep retry counters and deadlines in state. Each retry should have a clear category, maximum, and observable event.


In [ ]:
from hashlib import sha256

execution_ledger: dict[str, dict] = {}

def action_digest(proposal: dict) -> str:
    canonical = f"{proposal['incident_id']}|{proposal['action']}|{sorted(proposal['arguments'].items())}"
    return sha256(canonical.encode()).hexdigest()[:16]


def execute_once(proposal: dict, approval: dict) -> dict:
    """A simulated execution boundary with approval and idempotency checks."""
    if approval.get("decision") != "approve":
        return {"status": "blocked", "reason": "No verified approval"}
    key = action_digest(proposal)
    if execution_ledger.get(key):
        return {"status": "reused", "receipt": execution_ledger[key], "idempotency_key": key}
    receipt = {"action": proposal["action"], "service": proposal["arguments"]["service"], "receipt_id": f"receipt-{key}"}
    execution_ledger[key] = receipt
    return {"status": "executed", "receipt": receipt, "idempotency_key": key}

proposal = state["proposal"]
verified_approval = {"decision": "approve", "approver_id": "oncall-17", "action_digest": action_digest(proposal)}
first = execute_once(proposal, verified_approval)
second = execute_once(proposal, verified_approval)
{"first_attempt": first, "retry_with_same_key": second}


## 6. Step 4 — Stream progress and inspect the trajectory

A useful operator interface does not wait for a final answer. It can show a redacted stream of state transitions: evidence collected, confidence assessed, approval requested, decision received, and receipt recorded. Do not stream secrets, raw customer data, or hidden reasoning.

With LangGraph, `stream_events(..., version="v3")` can expose output, state values, message chunks, and interrupts. The [streaming guide](https://docs.langchain.com/oss/python/langgraph/streaming) documents the available projections.

```python
# Conceptual operator loop
stream = graph.stream_events(initial_state, config=config, version="v3")
for update in stream.values:
    publish_redacted_state(update)

if stream.interrupted:
    payload = stream.interrupts[0].value
    # Render a server-authorized review form. Do not expose arbitrary tool access.
    decision = await collect_verified_operator_decision(payload)
    stream = graph.stream_events(Command(resume=decision), config=config, version="v3")

final_state = stream.output
```

### Trace events for this incident

The small trace below is framework-neutral, but it is the shape to send to a tracing system: each event has a stable incident/thread identifier, an event type, and a safe payload.


In [ ]:
trace = [
    {"step": 1, "event": "node_started", "node": "collect_evidence"},
    {"step": 2, "event": "state_update", "evidence_count": len(state["evidence"])},
    {"step": 3, "event": "route", "value": branch, "confidence": state["confidence"]},
    {"step": 4, "event": "interrupt_requested", "action": proposal["action"]},
    {"step": 5, "event": "approval_verified", "operator": verified_approval["approver_id"]},
    {"step": 6, "event": "action_receipt", "receipt": first["receipt"]["receipt_id"]},
]
trace


## 7. Step 5 — Scope memory and isolate subgraphs

Memory is not automatically beneficial. A durable state record for this incident is useful; a cross-incident claim such as “checkout failures are usually Redis” can bias the next diagnosis.

| Data | Appropriate scope | Control |
| --- | --- | --- |
| current evidence and pending approval | thread/checkpoint | restrict to the incident; expire after retention period |
| approved operator communication preference | long-term store | validate source; offer deletion and audit writes |
| raw customer ticket details | secured case system | retrieve only when authorized; avoid copying into generic memory |
| unverified root-cause hypothesis | do not persist as a fact | retain as a labeled, reviewable hypothesis if needed |

A **subgraph** is useful when a bounded concern has its own state and lifecycle: for example, a read-only evidence collection subgraph that normalizes metrics, deployment history, and ticket facts. The parent graph should own the approval and execution boundary. Use explicit inputs/outputs so a subgraph cannot silently broaden privileges. See the [official subgraphs guide](https://docs.langchain.com/oss/python/langgraph/use-subgraphs).


In [ ]:
long_term_store: dict[tuple[str, str], dict] = {}

def store_preference(tenant_id: str, user_id: str, preference: str) -> None:
    """Allow-listed durable memory: preference, not diagnosis or secret evidence."""
    if preference not in {"brief_updates", "detailed_updates"}:
        raise ValueError("Only reviewed notification preferences are storable.")
    long_term_store[(tenant_id, user_id)] = {"notification_preference": preference}

store_preference("acme", "oncall-17", "brief_updates")
long_term_store


## 8. Production readiness checklist and exercises

### Before connecting real tools

- [ ] Every tool has a narrow schema, authorization check, timeout, and audit event.
- [ ] Graph state contains no API keys, raw credentials, or unrestricted personal data.
- [ ] Checkpoints use a durable, encrypted backend with tenant-scoped access controls and retention.
- [ ] Approval is server-side authenticated and bound to action digest, incident, role, and expiry.
- [ ] External effects are idempotent and can return a receipt after retries.
- [ ] Escalation is a first-class terminal route; “uncertain” does not become an action.
- [ ] Streams, logs, and traces redact sensitive values and have access controls.
- [ ] Evaluation includes approval bypass, duplicate execution, stale memory, timeout, and cross-tenant tests.

### Exercises

1. Add a `collect_more_evidence` route for confidence between 0.60 and 0.80. Give it a maximum of two attempts and an escalation condition.
2. Change the approval result to `approve`, `reject`, or `modify`. On `modify`, invalidate the old digest and return to proposal construction.
3. Add a tenant ID to state and ensure both the checkpoint configuration and action policy reject a mismatched tenant.
4. Write a test that calls `execute_once` three times after a simulated network response loss and proves only one receipt exists.
5. Build an evaluation record containing route selected, node count, approval latency, action receipt, and whether any forbidden action was attempted.

## References

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) — why the runtime focuses on explicit stateful orchestration.
- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — checkpointers, stores, thread scope, and durable backends.
- [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) — pause/resume semantics, approval patterns, and idempotency warning.
- [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming) — state, message, and interrupt event projections.
- [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) — composition and state-boundary guidance.

**Takeaway:** LangGraph does not replace good system design. It makes state transitions, recovery, human control, and execution boundaries explicit enough to design and test them.
